<a href="https://colab.research.google.com/github/talhanoor23/algorithmic-trading/blob/main/crypto_trading_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Get_Realtime_Data**

In [ ]:
# 1. Install TA-Lib C library from conda-forge
!curl -L \
'https://anaconda.org/conda-forge/libta-lib/0.4.0/download/linux-64/libta-lib-0.4.0-h166bdaf_1.tar.bz2' \
| tar xj -C /usr/lib/x86_64-linux-gnu/ lib --strip-components=1

# 2. Install TA-Lib Python wrapper via pip
!pip install TA-Lib

In [ ]:
import talib
print(len(talib.get_functions()), "functions available:")
print(talib.get_functions()[:5])   # shows first few indicators
print("\nGroups:", list(talib.get_function_groups().keys()))

In [ ]:
# !pip install --upgrade pip
# !pip install yfinance
# !pip uninstall -y statsmodels # Ensure a clean slate
# !pip install --upgrade statsmodels pandas_datareader
# !pip install numpy>=2.2.6 pandas_ta

In [ ]:
!pip uninstall -y numpy pandas statsmodels pandas_datareader pandas_ta numba scipy

In [ ]:
!pip install numpy==1.26.4 pandas==2.1.4 scipy==1.11.4 statsmodels==0.14.1 pandas_datareader==0.10.0 yfinance matplotlib

In [ ]:
# # Now proceed with the imports
# from statsmodels.regression.rolling import RollingOLS
# import pandas as pd
# # import pandas_ta as ta
# import yfinance as yf
# import numpy as np
# import pandas_datareader.data as pdr
# import matplotlib.pyplot as plt
# import statsmodels.api as sm
# import datetime as dt
# import warnings
# import websocket
# warnings.filterwarnings('ignore')

In [ ]:
from statsmodels.regression.rolling import RollingOLS
import pandas as pd
import numpy as np
import yfinance as yf
import pandas_datareader.data as pdr
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [ ]:
from pickle import TRUE
aroon_time_period = 3
amount = 1000

core_trade_amount = amount*0.80
core_trade_cond = True
core_trade_portfolio = 0

trade_amount = amount*0.20
amount_left = amount
portfolio = 0
investment = []

realtime_portfolio_value = []

In [ ]:
def buy(buying_amount, coin_price ):
    global amount_left, portfolio, investment
    quantity = buying_amount/coin_price
    amount_left -= buying_amount
    portfolio += quantity
    if investment == []:
        investment.append(buying_amount)
    else:
        investment.append(buying_amount)
        investment[-1] += investment[-2]


def sell(selling_amount, coin_price):
    global amount_left, portfolio, investment
    quantity = selling_amount/coin_price
    amount_left += selling_amount
    portfolio -= quantity
    investment.append(-selling_amount)
    investment[-1] += investment[-2]

In [ ]:
from talib import abstract

f = abstract
dir1 = dir(f)
print(dir1)

['ACCBANDS', 'ACOS', 'AD', 'ADD', 'ADOSC', 'ADX', 'ADXR', 'APO', 'AROON', 'AROONOSC', 'ASIN', 'ATAN', 'ATR', 'AVGDEV', 'AVGPRICE', 'BBANDS', 'BETA', 'BOP', 'CCI', 'CDL2CROWS', 'CDL3BLACKCROWS', 'CDL3INSIDE', 'CDL3LINESTRIKE', 'CDL3OUTSIDE', 'CDL3STARSINSOUTH', 'CDL3WHITESOLDIERS', 'CDLABANDONEDBABY', 'CDLADVANCEBLOCK', 'CDLBELTHOLD', 'CDLBREAKAWAY', 'CDLCLOSINGMARUBOZU', 'CDLCONCEALBABYSWALL', 'CDLCOUNTERATTACK', 'CDLDARKCLOUDCOVER', 'CDLDOJI', 'CDLDOJISTAR', 'CDLDRAGONFLYDOJI', 'CDLENGULFING', 'CDLEVENINGDOJISTAR', 'CDLEVENINGSTAR', 'CDLGAPSIDESIDEWHITE', 'CDLGRAVESTONEDOJI', 'CDLHAMMER', 'CDLHANGINGMAN', 'CDLHARAMI', 'CDLHARAMICROSS', 'CDLHIGHWAVE', 'CDLHIKKAKE', 'CDLHIKKAKEMOD', 'CDLHOMINGPIGEON', 'CDLIDENTICAL3CROWS', 'CDLINNECK', 'CDLINVERTEDHAMMER', 'CDLKICKING', 'CDLKICKINGBYLENGTH', 'CDLLADDERBOTTOM', 'CDLLONGLEGGEDDOJI', 'CDLLONGLINE', 'CDLMARUBOZU', 'CDLMATCHINGLOW', 'CDLMATHOLD', 'CDLMORNINGDOJISTAR', 'CDLMORNINGSTAR', 'CDLONNECK', 'CDLPIERCING', 'CDLRICKSHAWMAN', 'CDLRISEFA

In [ ]:
cdl_patterns = [func for func in dir1 if func.startswith('CDL')]
print(cdl_patterns)

In [ ]:
import websocket
import json

closes = []
highs = []
lows = []
volumes = []

def on_message(ws, message):
    global closes, highs, lows, core_trade_cond, core_trade_amount, trade_amount, core_trade_portfolio, amount_left, portfolio, investment

    data = json.loads(message)

    # Check if the message contains kline data
    if "data" not in data or not isinstance(data["data"], list):
        return

    for kline in data["data"]:
        candle_closed = kline.get("confirm", False)

        if candle_closed:
            closes.append(float(kline["close"]))
            highs.append(float(kline["high"]))
            lows.append(float(kline["low"]))
            volumes.append(float(kline["volume"]))

            inputs = {
                'open': np.array(closes),
                'high': np.array(highs),
                'low': np.array(lows),
                'close': np.array(closes),
                'volume': np.array(volumes)
            }
            print(inputs)


            if core_trade_cond:
              buy(core_trade_amount, closes[-1])
              core_trade_portfolio += core_trade_amount/closes[-1]
              print(f"__ we bought ${core_trade_amount} worth of bitcoin __")
              core_trade_cond = False
            # else:
            #   sell(trade_amount, closes[-1])


            indicators = []
            for method in cdl_patterns:
              indicator = getattr(f, method)(inputs)
              indicators.append(indicator[-1])
            avg_indicators = np.mean(indicators)



            if avg_indicators >= 10:
              amt = trade_amount
            elif avg_indicators <= -10:
              amt = -trade_amount
            else:
              amt = avg_indicators * 10
            port_value = (portfolio * closes[-1]) - (core_trade_portfolio * closes[-1])
            trade_amt = amt - port_value


            RT_portfolio_value = amount_left + (portfolio * closes[-1])
            realtime_portfolio_value.append(float(RT_portfolio_value))


            print(f"indicators : {indicators}")
            print(f"avg_indicators : {avg_indicators}")
            print(f"core_trade_portfolio : {core_trade_portfolio}")
            print("trade_amount", trade_amt)
            print("port_value", port_value)
            print(f"realtime_portfolio_value : {RT_portfolio_value}")
            print(f"investment : {portfolio*closes[-1]}")
            print(f"amount_left : {amount_left}")

            print("--------------------------------------------------------------------------------")

            if trade_amt > 0:
              buy(trade_amt, closes[-1])
              print(f"__ we bought ${trade_amt} worth of bitcoin __")
            elif trade_amt < 0:
              sell(abs(trade_amt), closes[-1])
              print(f"__ we sold ${abs(trade_amt)} worth of bitcoin __")

            print("--------------------------------------------------------------------------------")



def on_open(ws):
    # Subscribe to BTCUSDT 1-minute candlestick
    params = {
        "op": "subscribe",
        "args": ["kline.1.BTCUSDT"]
    }
    ws.send(json.dumps(params))

def on_close(ws, close_status_code, close_msg):
    print("WebSocket closed:", close_status_code, close_msg)

socket = "wss://stream.bybit.com/v5/public/linear"

ws = websocket.WebSocketApp(socket,
                            on_message=on_message,
                            on_open=on_open,
                            on_close=on_close)

In [ ]:
ws.run_forever()
#aroon timeperiod is set to 3, so it shows after 3 interval. greater the timepriod, prediction is more accurate or have high accuracy like in models, but here we deal with aroon indicators.